## Install packages

In [101]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-chroma \
    langchain-text-splitters \
    langchain-classic \
    pypdf \
    chromadb \
    python-dotenv \
    pandas \
    tiktoken

## Load environment and basic settings

In [102]:
import os
import getpass
import shutil
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    api_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = api_key

data_dir = "data"
#pdf_path = Path(f"{data_dir}/dotcom-secrets.pdf")
pdf_path = Path("data/MA 5.0.x Detailed Design and workflow Document.pdf")
DB_DIR = Path(f"{data_dir}/db")
VECTOR_STORE_PATH = Path(f"{DB_DIR}/vector_db/pdf_vector_db_chroma")
EMBEDDING_CACHE_PATH = Path(f"{DB_DIR}/embedding_cache/pdf_embeddings_cache")
ANSWER_CACHE_PATH = Path(f"{DB_DIR}/answer_cache")
ANSWER_CACHE_DB = Path(f"{DB_DIR}/answer_cache/pdf_answers_cache")

CSV_FILE_PATH = Path(f"{DB_DIR}/pdf_chunks.csv")
shutil.rmtree(DB_DIR, ignore_errors=True)

VECTOR_STORE_PATH.mkdir(parents=True, exist_ok=True)
EMBEDDING_CACHE_PATH.mkdir(parents=True, exist_ok=True)
ANSWER_CACHE_PATH.mkdir(parents=True, exist_ok=True)


assert pdf_path.exists(), f"PDF file not found at: {pdf_path}"
print(f"PDF file found at: {pdf_path}")

PDF file found at: data/MA 5.0.x Detailed Design and workflow Document.pdf


## Load the PDF

PyPDFLoader loads a PDF into LangChain Document objects. LangChain docs mention that this loader commonly creates one Document per PDF page and includes metadata like source and page number.

In [103]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(str(pdf_path))
document = loader.load_and_split()

print(f"Loaded {len(document)} chunks from the PDF.")
print(f"First chunk preview:\n{document[0].page_content[:500]}")
print(f"Metadata of first chunk:\n{document[0].metadata}")

Loaded 184 chunks from the PDF.
First chunk preview:
John, DianaX 
INTEL SECURITY GROUP        
McAfee Agent 5.0 Design Document
Metadata of first chunk:
{'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2021-03-03T03:44:09-08:00', 'title': 'McAfee Agent 5.0 Design Document', 'author': 'John, DianaX', 'moddate': '2021-03-03T03:44:09-08:00', 'source': 'data/MA 5.0.x Detailed Design and workflow Document.pdf', 'total_pages': 195, 'page': 0, 'page_label': '1'}


## Split PDF into chunks

For books, page-level chunks are usually too large. We split into smaller overlapping chunks. LangChain docs show RecursiveCharacterTextSplitter with overlap because overlap helps avoid losing context between chunk boundaries.

In [104]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

splits = text_splitter.split_documents(document)

print(f"Total chunks created: {len(splits)}")
print(f"First chunk preview:\n{splits[0].page_content[:500]}")
print(f"Metadata of first chunk:\n{splits[0].metadata}")

Total chunks created: 347
First chunk preview:
John, DianaX 
INTEL SECURITY GROUP        
McAfee Agent 5.0 Design Document
Metadata of first chunk:
{'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2021-03-03T03:44:09-08:00', 'title': 'McAfee Agent 5.0 Design Document', 'author': 'John, DianaX', 'moddate': '2021-03-03T03:44:09-08:00', 'source': 'data/MA 5.0.x Detailed Design and workflow Document.pdf', 'total_pages': 195, 'page': 0, 'page_label': '1', 'start_index': 0}


## Create a DataFrame of chunks (Optional)

This useful for debugging. You can inspect which page/chunk is going into the vector DB.

In [105]:
import pandas as pd
chunks_df= pd.DataFrame([
{
    "chunk_id": i,
        "source": d.metadata.get("source"),
        "page": d.metadata.get("page", 0) + 1,
        "start_index": d.metadata.get("start_index"),
        "text_preview": d.page_content[:500]
}
for i, d in enumerate(splits)
])

chunks_df.head()

,chunk_id,source,page,start_index,text_preview
0,0,data/MA 5.0.x Detailed Design and workflow Doc...,1,0,"John, DianaX \nINTEL SECURITY GROUP \nM..."
1,1,data/MA 5.0.x Detailed Design and workflow Doc...,2,0,Abstract \nThis document contains the detailed...
2,2,data/MA 5.0.x Detailed Design and workflow Doc...,3,0,Contents \nIntroduction .........................
3,3,data/MA 5.0.x Detailed Design and workflow Doc...,3,821,McAfee Agent Architecture .......................
4,4,data/MA 5.0.x Detailed Design and workflow Doc...,3,1640,Message reference counting ......................


In [106]:
#chunks_df.to_csv(f"{DB_DIR}/ma.csv", index=False)
chunks_df.to_csv(CSV_FILE_PATH, index=False)
ANSWER_CACHE_DB
print("Saved chunk map to ma.csv")

Saved chunk map to ma.csv


## Create embeddings with embedding cache

Purpose: avoid paying/recomputing embeddings again and again for the same PDF chunks. LangChain documents CacheBackedEmbeddings, which stores embeddings in a key-value store and can use LocalFileStore for local persistent storage.

In [107]:
from langchain_openai import OpenAIEmbeddings

# LangChain has moved some cache/storage classes across versions.
# This fallback keeps the notebook safer.
try:
    from langchain_classic.embeddings import CacheBackedEmbeddings
    from langchain_classic.storage import LocalFileStore
except ImportError:
    from langchain.embeddings import CacheBackedEmbeddings
    from langchain.storage import LocalFileStore

underlying_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

embedding_store = LocalFileStore(EMBEDDING_CACHE_PATH)

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    embedding_store,
    namespace="text-embedding-3-small",
    query_embedding_cache=True
)

print("Embedding cache is ready.")

Embedding cache is ready.


## Create or load Chroma vector database

In [108]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="dotcom_secrets_book",
    embedding_function=cached_embeddings,
    persist_directory=VECTOR_STORE_PATH
)

# Add documents only if DB is empty
existing_count = vector_store._collection.count()

if existing_count == 0:
    print("Vector DB is empty. Adding chunks...")
    vector_store.add_documents(splits)
    print("Chunks added:", len(splits))
else:
    print("Vector DB already has documents:", existing_count)

Vector DB already has documents: 347


## Create Retriever

In [109]:
retriever = vector_store.as_retriever(
    search_type="mmr", # MMR gives diverse results by balancing relevance and diversity
    search_kwargs={
        "k": 5, 
        "fetch_k": 20
        }
)

print("Retriever is set up and ready to use.")

Retriever is set up and ready to use.


## Create RAG chatbot with chat memory

Here we will use a history-aware retriever, so follow-up questions like “explain that again” or “give example of this” can still use conversation context.

I am avoiding the older ConversationalRetrievalChain pattern because the LangChain reference marks it as deprecated and points users toward create_retrieval_chain.

In [110]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 1. Prompt to rewrite follow-up questions into standalone questions
contextualize_q_system_prompt = """
Given a chat history and the latest user question, rewrite the latest question
as a standalone question that can be understood without chat history.

Do not answer the question. Only rewrite it if needed.
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)

# 2. Prompt to answer using retrieved book context
qa_system_prompt = """
You are a helpful RAG chatbot answering questions from the provided PDF book context.

Rules:
1. Answer only from the retrieved context.
2. If the answer is not present in the context, say: "I could not find this in the provided PDF context."
3. Do not fabricate.
4. Keep the answer clear and practical.
5. When useful, give bullet points.
6. Do not quote large copyrighted passages. Summarize instead.

Context:
{context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(
    llm,
    qa_prompt
)

rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

chat_history = []

print("RAG chatbot chain is ready.")

RAG chatbot chain is ready.


## Add answer cache using SQLite

This is your second cache.

Purpose: if you ask the same question again, the chatbot returns the stored answer instantly instead of calling the LLM again.

In [111]:
import sqlite3
import hashlib
import json
from datetime import datetime

def init_answer_cache(db_path=ANSWER_CACHE_DB):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS answer_cache (
            cache_key TEXT PRIMARY KEY,
            question TEXT,
            answer TEXT,
            sources TEXT,
            created_at TEXT
        )
    """)
    conn.commit()
    conn.close()

def normalize_question(question: str) -> str:
    return " ".join(question.lower().strip().split())

def make_cache_key(question: str, chat_history, include_history=False) -> str:
    """
    If include_history=True, the same question in different conversation contexts
    will have different cache keys.
    """
    normalized_q = normalize_question(question)
    
    history_text = ""
    if include_history:
        last_turns = chat_history[-6:]
        history_text = "\n".join([
            f"{type(m).__name__}: {m.content}" for m in last_turns
        ])
    
    raw_key = normalized_q + "\n" + history_text
    return hashlib.sha256(raw_key.encode("utf-8")).hexdigest()

def get_cached_answer(cache_key, db_path=ANSWER_CACHE_DB):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(
        "SELECT answer, sources FROM answer_cache WHERE cache_key = ?",
        (cache_key,)
    )
    row = cur.fetchone()
    conn.close()
    
    if row:
        answer, sources_json = row
        sources = json.loads(sources_json)
        return {
            "answer": answer,
            "sources": sources
        }
    return None

def save_cached_answer(cache_key, question, answer, sources, db_path=ANSWER_CACHE_DB):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        INSERT OR REPLACE INTO answer_cache
        (cache_key, question, answer, sources, created_at)
        VALUES (?, ?, ?, ?, ?)
    """, (
        cache_key,
        question,
        answer,
        json.dumps(sources),
        datetime.now().isoformat()
    ))
    conn.commit()
    conn.close()
print(ANSWER_CACHE_DB)
init_answer_cache()
print("Answer cache is ready.")

data/db/answer_cache/pdf_answers_cache
Answer cache is ready.


## Helper function for source citations

In [112]:
def format_sources(context_docs):
    sources = []
    
    for doc in context_docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        
        if isinstance(page, int):
            page = page + 1
        
        sources.append({
            "source": source,
            "page": page,
            "start_index": doc.metadata.get("start_index")
        })
    
    # remove duplicates
    unique_sources = []
    seen = set()
    
    for s in sources:
        key = (s["source"], s["page"], s["start_index"])
        if key not in seen:
            seen.add(key)
            unique_sources.append(s)
    
    return unique_sources

## Final chatbot function

In [113]:
def ask_book(question: str, use_cache=True, include_history_in_cache=False):
    global chat_history
    
    cache_key = make_cache_key(
        question,
        chat_history,
        include_history=include_history_in_cache
    )
    
    if use_cache:
        cached = get_cached_answer(cache_key)
        if cached:
            print("Returned from answer cache.\n")
            return cached
    
    response = rag_chain.invoke({
        "input": question,
        "chat_history": chat_history
    })
    
    answer = response["answer"]
    context_docs = response.get("context", [])
    sources = format_sources(context_docs)
    
    # update memory
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    
    # save answer cache
    if use_cache:
        save_cached_answer(
            cache_key=cache_key,
            question=question,
            answer=answer,
            sources=sources
        )
    
    return {
        "answer": answer,
        "sources": sources
    }

## We can Ask questions from our PDF

In [117]:
result = ask_book("what is updater?")

print(result["answer"])
print("\nSources:")
for s in result["sources"]:
    print(s)

The Updater service in McAfee Agent 5.0 is a manageability service responsible for handling update requests and creating update tasks for clients. Here are the key points about the Updater service:

- **Functionality**:
  - Manages update requests and creates tasks for both managed and unmanaged McAfee products.
  - Routes UI information, such as progress updates during an update, to products that have subscribed for notifications.

- **Update Items**:
  - Updates various components from a designated repository, including:
    - Patch releases
    - Legacy product plug-in (.DLL) files
    - Service pack releases
    - SuperDAT (SDAT*.EXE) packages
    - Supplemental detection definitions (ExtraDAT files)
    - Detection definition (DAT) files
    - Anti-virus engines

- **Update Request Types**:
  - Supports two types of update requests:
    - **Start Request**: Initiates the update process, checks for the nearest repository, and ensures the update request handler is available.
    - *